In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
## going to test the new setup
from nba import NBAbase, NBAetl, NBAmodels, NBAdata
#from dev import NBAdata
from betting import odds
import pandas as pd
import seaborn as sns
import sqlite3
import shutil
import numpy as np
from betting import funcs
odds = funcs.odds()
import matplotlib.pyplot as plt

In [10]:
q = '''CREATE TABLE IF NOT EXISTS predictions (
    player_id       TEXT,
    date            TEXT,
    over_under      TEXT,
    number          REAL,
    model_prob      REAL,
    FanDuels         REAL,
    DraftKings      REAL,
    theScore_Bet         REAL,
    ev_fanduel      REAL,
    ev_draftkings   REAL,
    ev_espnbet      REAL,
    bet_book        TEXT,
    final_line      REAL,
    bet_amount      REAL,
    result          REAL,
    PRIMARY KEY (player_id, date, over_under, number)
);'''

In [7]:
etl = NBAetl.etl()
data = NBAdata.data()
odds = funcs.odds()

In [12]:
etl.cur.execute(q)
etl.conn.commit()

In [15]:
#run file
#get yesterday's data and load
yst = (dt.datetime.today() + pd.to_timedelta(-1,unit='day')).strftime(format='%Y-%m-%d')
gids = etl.get_games(yst,yst)
print('Updating for {}'.format(yst))
etl.update_player_log([yst])
time.sleep(np.random.randint(5,15))
try:
    etl.update_shots_allowed([yst])
except:
    print('shots not updated yet')
time.sleep(np.random.randint(5,15)) 
etl.update_teamLog(gids.GAME_ID.unique())

#get the model and run the data through the pipeline
threes = NBAmodels.models('threes')

td = data.threes_pipe(threes.data)

td = data.clean_na(td)
td = td[td.game_date==(dt.datetime.today()).strftime('%Y-%m-%d')]
td = threes.standRobust_scaler(td)
#get predictions
preds= threes.model.predict(sm.add_constant(td.filter(threes.features),has_constant='add'))

#name alterations
idInfo = threes.data[threes.data.game_date==dt.datetime.today().strftime('%Y-%m-%d')][['name','team','game_id']]
idInfo.name = np.where(idInfo.name.str.contains('\.'),idInfo.name.str.replace('.',''),idInfo.name)
idInfo.name = np.where(idInfo.name.str.contains('ë'),idInfo.name.str.replace('ë','e'),idInfo.name)
idInfo.name = np.where(idInfo.name =='Derrick Jones Jr','Derrick Jones',idInfo.name)
idInfo.name = np.where(idInfo.name =='KyShawn George','Kyshawn George',idInfo.name)
idInfo.name = np.where(idInfo.name =='Isaiah Stewart','Isaiah Stewart II',idInfo.name)
idInfo.name = np.where(idInfo.name== 'Ronald Holland II','Ron Holland',idInfo.name)

#odds pull
overs = odds.oddsTable(preds,idInfo)


#%2cespnbet
def fetch_odds(market):
    url = odds.market_vars.get(market).get('url')
    df = pd.DataFrame()
    events,akey = odds.oddsData(odds.nbaEvents,usePaid=True)
    for event in events:
        r = requests.get(url.format(event,akey))
        game = r.json()
        for key in game.get('bookmakers'):
            bk = key.get('title')
            for mrkt in key.get('markets'):
                temp = pd.DataFrame(mrkt.get('outcomes'))
                temp.columns = ['over_under','name','price',odds.market_vars.get(market).get('col_name')]
                temp['book'] = bk
                df = pd.concat([temp,df])   
    odf = df.pivot_table(index=['name',odds.market_vars.get(market).get('col_name'),'over_under'],columns=['book']).reset_index()
    odf.columns = [col[1] if col[1]!= '' else col[0] for col in odf.columns]
    return odf
#odds name updates
odf.name = np.where(odf.name=='Herb Jones','Herbert Jones',odf.name)
odf.name = np.where(odf.name =='Alex Sarr','Alexandre Sarr',odf.name)
odf.name = np.where(odf.name =='Tristan da Silva','Tristan Da Silva',odf.name)
odf.name = np.where(odf.name.str.contains('\.'),odf.name.str.replace('.',''),odf.name)

#create the data table view
def bet_table(overs,odf,val_col):
    
    final = odf.merge(overs,how='left',on=['name',val_col,'over_under'])
    final['prob'] = np.where(final.value<0, round(abs(final.value) / (abs(final.value) + 100),4), round(100/(final.value +100),4))
    final['DKEV'] = ['{:.2%}'.format(odds.ev(p, odd)) for p,odd in zip(final.prob,final.DraftKings.replace(0,1))]
    final['FanDuelEV'] = ['{:.2%}'.format(odds.ev(p, odd))  for p,odd in zip(final.prob,final.FanDuel.replace(0,1))]
    #final['BetMGMEV'] = ['{:.2%}'.format(odds.ev(p, odd))  for p,odd in zip(final.prob,final.BetMGM)]
    final['espnEV'] = ['{:.2%}'.format(odds.ev(p, odd))  for p,odd in zip(final.prob,final['theScore Bet'])]
    #first row needs to be the tiebreaker row
    final['FanDuelAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.FanDuelKelly.values]
    final['DraftKingsAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.DKKelly.values]
    #final['BetMGMAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.BetMGMKelly.values]
    final['theScore BetAmount'] = [round(x * odds.budget * odds.kellyVal,2) for x in final.espnKelly]
    return final


NameError: name 'dt' is not defined

In [18]:
# run_daily.py

import datetime as dt

def run_model(date = dt.datetime.today().strftime('%Y-%m-%d')):
    threes = NBAmodels.models('threes')
    td = data.threes_pipe(threes.data)
    td = data.clean_na(td)
    td = td[td.game_date == date]
    td = threes.standRobust_scaler(td)
    preds = threes.model.predict(sm.add_constant(td.filter(threes.features), has_constant='add'))
    idInfo = threes.data[threes.data.game_date == date][['name','team','game_id']]
    idInfo.name = standardize_names(idInfo.name)
    return odds.oddsTable(preds, idInfo)

odf = odds.fetch_odds('threes')


def write_predictions(final, conn):
    today = dt.datetime.today().strftime('%Y-%m-%d')
    final['date'] = today
    final['bet_book'] = None
    final['final_line'] = None
    final['bet_amount'] = None
    final['result'] = None
    final.rename(columns={'threesMade':'number','name':'player_id'}, inplace=True)
    final[['player_id','date','over_under','number','prob',
           'DraftKings','FanDuel','ESPN Bet',
           'ev_draftkings','ev_fanduel','ev_espnbet',
           'bet_book','final_line','bet_amount','result']].to_sql('predictions', conn, if_exists='append', index=False)



if __name__ == '__main__':
    yst = (dt.datetime.today() + pd.to_timedelta(-1, unit='day')).strftime('%Y-%m-%d')
    conn = sqlite3.connect('nba.db')
    run_data_update(yst)
    overs = run_model()
    odf, o = fetch_odds()
    final = build_final(overs, odf, o)
    write_predictions(final, conn)

Free is out
{'Date': 'Thu, 19 Mar 2026 11:14:12 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '535', 'Connection': 'keep-alive', 'X-Requests-Used': '923', 'X-Requests-Remaining': '19077', 'X-Requests-Last': '0', 'vary': 'Accept-Encoding', 'content-encoding': 'gzip', 'Apigw-Requestid': 'ad8UvgkKIAMEMXg='}


NameError: name 'run_data_update' is not defined

In [19]:
odf

,name,threesMade,over_under,DraftKings,FanDuel,theScore Bet
0,A.J. Green,0.5,Over,-1420.0,-1200.0,NaN
1,A.J. Green,1.5,Over,-264.0,-265.0,-275.0
2,A.J. Green,2.5,Over,107.0,109.0,105.0
3,A.J. Green,2.5,Under,-141.0,-144.0,-140.0
4,A.J. Green,3.5,Over,255.0,250.0,240.0
...,...,...,...,...,...,...
568,Will Riley,1.5,Under,-118.0,-125.0,-125.0
569,Will Riley,2.5,Over,270.0,260.0,250.0
570,Will Riley,3.5,Over,720.0,600.0,700.0
571,Will Riley,4.5,Over,1680.0,1300.0,1400.0
